In [1]:
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

v:\Customer Support Agent\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from core.graph import build_graph

graph = build_graph()

v:\Customer Support Agent\app\rag\retriever.py:5: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  _embeddings = HuggingFaceBgeEmbeddings(
v:\Customer Support Agent\app\rag\retriever.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  _vectorstore = Chroma(


In [3]:
with open("../evaluation/evaluation.json") as f:
    eval_data = json.load(f)

print(f"Loaded {len(eval_data)} evaluation samples")

Loaded 52 evaluation samples


In [ ]:
results = []

for item in tqdm(eval_data):
    output = graph.invoke({"user_query": item["query"]})

    results.append({
        "query":item["query"],
        "expected_action": item["expected_action"],
        "predicted_action":output.get("action"),
        "answer":output.get("answer"),
        "reference_answer": item["reference_answer"],
        "confidence":output.get("answer_confidence", 0.0),
        "retrived_docs": item.get("reference_answer",""),
        "expected_docs":item.get("expected_docs",[])
    })


df = pd.DataFrame(results)
df.head()

100%|██████████| 52/52 [09:58<00:00, 11.50s/it]


,query,expected_action,predicted_action,answer,confidence,retrived_docs,expected_docs
0,refund not received after cancellation,answer,answer,**Refund after trial cancellation**\n\n1. **Ch...,0.85,Refunds are usually processed within 5 to 7 bu...,"[refund_status.md, refund_processing_time.md]"
1,charged after subscription cancellation,escalate,answer,If you were charged after cancelling your subs...,0.92,,[subscription_charged_after_cancellation.md]
2,how long does it take to get refund,answer,answer,Refunds are typically processed within **5‑7 b...,0.95,Refund processing typically takes between 5 an...,[refund_processing_time.md]
3,refund declined without reason,escalate,answer,Check the status of your refund request in you...,0.85,,[refund_declined.md]
4,can I get refund after trial cancellation,answer,answer,"Yes. After you cancel your trial, check the ca...",0.85,Refunds after trial cancellation depend on whe...,[refund_after_trial_cancellation.md]


In [5]:
df["routing_correct"] = df["expected_action"] == df["predicted_action"]

routing_accuracy = df["routing_correct"].mean()

missed_escalation = (
    (df.expected_action == "escalate") &
    (df.predicted_action != "escalate")
).mean()

false_escalation = (
    (df.expected_action != "escalate") &
    (df.predicted_action == "escalate")
).mean()

routing_accuracy, missed_escalation, false_escalation

(np.float64(0.5384615384615384),
 np.float64(0.17307692307692307),
 np.float64(0.19230769230769232))

In [16]:
def doc_hit(retrieved_docs, expected_docs):
    hits = 0
    for exp in expected_docs:
        if any(exp.lower() in d.lower() for d in retrieved_docs):
            hits +=1
    return hits / max(len(expected_docs), 1)

In [ ]:
def recall_at_k(retrieved, expected):
    if not expected:
        return 1.0  # nothing expected
    hits = set(retrieved) & set(expected)
    return len(hits) / len(expected)

df["recall_at_k"] = df.apply(
    lambda r: recall_at_k(r["retrieved_docs"], r["expected_docs"]),
    axis=1
)


np.float64(0.0)

In [20]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def semantic_similarity(a, b):
    if not a or not b:
        return 0.0
    emb = embedder.encode([a, b])
    return cosine_similarity([emb[0]], [emb[1]])[0][0]

df["answer_similarity"] = df.apply(
    lambda r: semantic_similarity(r.answer, r.reference_answer),
    axis=1
)

df["answer_similarity"].mean()

AttributeError: 'Series' object has no attribute 'reference_answer'

In [ ]:
possible_hallucination = (answer exists) & (recall_at_k < 0.3)


df["hallucination"] = (
    (df["predicted_action"] == "answer") &
    (df["recall_at_k"] < 0.5)
)



np.float64(0.5576923076923077)

In [22]:
df["correct_answer"] = df.answer_similarity > 0.7

confidence_bins = pd.cut(
    df.confidence,
    bins=[0, 0.4, 0.7, 1.0],
    labels=["low", "medium", "high"]
)

df.groupby(confidence_bins)["correct_answer"].mean()


AttributeError: 'DataFrame' object has no attribute 'answer_similarity'